# 04 - Cross-Product LGD Integration

**One Integrated LGD Framework - Australian Bank-Style Portfolio View**

This notebook standardises product comparisons across mortgage, commercial, development, CRE investment, residual stock, land/subdivision, bridging, and mezz/2nd mortgage.

| Step | Description |
|---|---|
| 1 | Build cross-product segmentation taxonomy |
| 2 | Assemble weighted LGD metrics by product |
| 3 | Compare weighted LGD, downturn sensitivity, and recovery time |
| 4 | Analyse portfolio mix and contribution |
| 5 | Create transparent risk ranking |


In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_generation import generate_all_datasets
from src.lgd_calculation import run_full_pipeline
from src.reproducibility import set_global_seed

set_global_seed(54)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 120)
pd.set_option('display.float_format', '{:.4f}'.format)

PROJECT_ROOT = Path('..').resolve()
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
TABLE_DIR = OUTPUT_DIR / 'tables'
FIGURE_DIR = OUTPUT_DIR / 'figures'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SHOW_PLOTS = os.environ.get('LGD_NOTEBOOK_SHOW_PLOTS', '0') == '1'
print(f"Interactive plot display enabled: {SHOW_PLOTS}")

PRODUCT_ORDER = [
    'Residential Mortgage',
    'Commercial Cashflow',
    'Development Finance',
    'CRE Investment',
    'Residual Stock',
    'Land / Subdivision',
    'Bridging Loan',
    'Mezzanine / 2nd Mortgage',
]


## Governance Baseline (Pre-Step-3)

- **Fallback hierarchy (standard policy):** use observed workout inputs first, then approved proxy inputs, then conservative fallback with explicit disclosure.
- **Proxy logic (standard policy):** transparent proxy assumptions are permitted where observed workout tapes are unavailable.
- **Cure / resolution treatment:** cure is applied where product-appropriate (especially mortgage); recovery waterfall and collateral realisation logic are used for non-mortgage secured products.
- **Discount-rate policy:** `discount_rate = max(contract_rate_proxy, cost_of_funds_proxy)`.
- **Calibration status (standard policy):** this cross-product view is a portfolio-project proxy baseline, **not** internally calibrated to real workout outcomes.
- **Output standard:** compare `lgd_base`, `lgd_downturn`, `lgd_final`, downturn sensitivity, and base recovery-time metric across products using EAD-weighted definitions.


## Validation Context (Simulated)

This comparison is a **simulated validation view** using transparent proxy data and assumptions. It is designed for portfolio communication and interview discussion, not production model approval.


## 1. Cross-Product Segmentation Table


In [ ]:
segmentation_table = pd.DataFrame([
    {
        'product': 'Residential Mortgage',
        'repayment_source': 'Borrower income, cure, property sale, LMI where eligible',
        'security': 'First-ranking residential mortgage (with optional LMI support)',
        'drivers': 'LVR, LMI, cure rate, arrears proxy, foreclosure/workout path',
    },
    {
        'product': 'Commercial Cashflow',
        'repayment_source': 'Operating cashflow plus collateral/workout recoveries',
        'security': 'PPSR/GSA and other business security packages',
        'drivers': 'DSCR/ICR, leverage, security coverage, industry risk, workout strategy',
    },
    {
        'product': 'Development Finance',
        'repayment_source': 'Project completion and sales/refinance exit',
        'security': 'Project assets and assignment of project cashflows/presales',
        'drivers': 'GRV decline, completion %, cost-to-complete, sell-through delay, presales',
    },
    {
        'product': 'CRE Investment',
        'repayment_source': 'Rental income durability, refinance or forced sale',
        'security': 'Stabilised investment property security',
        'drivers': 'LVR, DSCR, WALE, vacancy, tenant concentration, cap-rate expansion',
    },
    {
        'product': 'Residual Stock',
        'repayment_source': 'Sale of completed but unsold units',
        'security': 'Residual completed stock inventory',
        'drivers': 'Unsold units, absorption speed, discount-to-clear, holding costs, time to sale',
    },
    {
        'product': 'Land / Subdivision',
        'repayment_source': 'Land lot disposal over extended sale horizon',
        'security': 'Land/subdivision collateral with low carry income',
        'drivers': 'Zoning/stage, liquidity risk, market depth, value haircut, time to sell',
    },
    {
        'product': 'Bridging Loan',
        'repayment_source': 'Planned refinance or sale exit',
        'security': 'Short-tenor property-backed bridge security',
        'drivers': 'Exit type, exit certainty, valuation risk, time to exit, failed-exit risk',
    },
    {
        'product': 'Mezzanine / 2nd Mortgage',
        'repayment_source': 'Residual collateral value after senior debt repayment',
        'security': 'Subordinated lien / second-ranking security',
        'drivers': 'Total LVR, attachment point, subordination, collateral value decline',
    },
])

display(segmentation_table)
segmentation_table.to_csv(TABLE_DIR / 'cross_product_segmentation_table.csv', index=False)


## 2. Build Integrated Product Comparison Inputs


In [ ]:
datasets = generate_all_datasets()
core_results = run_full_pipeline(datasets)


def _safe_weighted(df, value_col, ead_col):
    if value_col not in df.columns or ead_col not in df.columns:
        return np.nan
    x = pd.to_numeric(df[value_col], errors='coerce')
    w = pd.to_numeric(df[ead_col], errors='coerce').fillna(0.0)
    mask = x.notna() & w.notna()
    if mask.sum() == 0 or w.loc[mask].sum() <= 0:
        return np.nan
    return float((x.loc[mask] * w.loc[mask]).sum() / w.loc[mask].sum())


def _summary_from_core(product_label, key):
    df = core_results[key]['loans_with_overlays'].copy()
    base_col = 'lgd_base' if 'lgd_base' in df.columns else 'realised_lgd'
    downturn_col = 'lgd_downturn' if 'lgd_downturn' in df.columns else base_col
    final_col = 'lgd_final' if 'lgd_final' in df.columns else downturn_col
    recovery_col = 'workout_months' if 'workout_months' in df.columns else None

    return {
        'product': product_label,
        'loan_count': len(df),
        'total_ead': float(pd.to_numeric(df['ead'], errors='coerce').fillna(0.0).sum()),
        'ead_weighted_lgd_base': _safe_weighted(df, base_col, 'ead'),
        'ead_weighted_lgd_downturn': _safe_weighted(df, downturn_col, 'ead'),
        'ead_weighted_lgd_final': _safe_weighted(df, final_col, 'ead'),
        'mean_recovery_time_months': float(pd.to_numeric(df[recovery_col], errors='coerce').mean()) if recovery_col else np.nan,
        'data_source': 'core_pipeline',
        'coverage_status': 'available',
        'notes': 'explicit final LGD from core APRA overlay pipeline',
    }


def _summary_from_loan_csv(product_label, filename, ead_col, base_col, downturn_col, final_col, recovery_col):
    path = TABLE_DIR / filename
    if not path.exists():
        return {
            'product': product_label,
            'loan_count': 0,
            'total_ead': 0.0,
            'ead_weighted_lgd_base': np.nan,
            'ead_weighted_lgd_downturn': np.nan,
            'ead_weighted_lgd_final': np.nan,
            'mean_recovery_time_months': np.nan,
            'data_source': filename,
            'coverage_status': 'missing_output',
            'notes': 'run product notebook to generate output table',
        }

    df = pd.read_csv(path)
    required_cols = [ead_col, base_col, downturn_col, final_col, recovery_col]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        return {
            'product': product_label,
            'loan_count': int(len(df)),
            'total_ead': float(pd.to_numeric(df[ead_col], errors='coerce').fillna(0.0).sum()) if ead_col in df.columns else np.nan,
            'ead_weighted_lgd_base': np.nan,
            'ead_weighted_lgd_downturn': np.nan,
            'ead_weighted_lgd_final': np.nan,
            'mean_recovery_time_months': np.nan,
            'data_source': filename,
            'coverage_status': 'invalid_output',
            'notes': f"missing required columns: {', '.join(missing_cols)}",
        }

    return {
        'product': product_label,
        'loan_count': int(len(df)),
        'total_ead': float(pd.to_numeric(df[ead_col], errors='coerce').fillna(0.0).sum()),
        'ead_weighted_lgd_base': _safe_weighted(df, base_col, ead_col),
        'ead_weighted_lgd_downturn': _safe_weighted(df, downturn_col, ead_col),
        'ead_weighted_lgd_final': _safe_weighted(df, final_col, ead_col),
        'mean_recovery_time_months': float(pd.to_numeric(df[recovery_col], errors='coerce').mean()),
        'data_source': filename,
        'coverage_status': 'available',
        'notes': f'explicit final LGD from {final_col}; recovery time from {recovery_col}',
    }


rows = [
    _summary_from_core('Residential Mortgage', 'mortgage'),
    _summary_from_core('Commercial Cashflow', 'commercial'),
    _summary_from_core('Development Finance', 'development'),
    _summary_from_loan_csv(
        'CRE Investment',
        'cre_investment_loan_level_output.csv',
        ead_col='ead',
        base_col='lgd_base',
        downturn_col='lgd_downturn',
        final_col='lgd_final',
        recovery_col='time_to_recovery_months_base',
    ),
    _summary_from_loan_csv(
        'Residual Stock',
        'residual_stock_loan_level_output.csv',
        ead_col='ead',
        base_col='lgd_base',
        downturn_col='lgd_downturn',
        final_col='lgd_final',
        recovery_col='time_to_sale_base',
    ),
    _summary_from_loan_csv(
        'Land / Subdivision',
        'land_subdivision_loan_level_output.csv',
        ead_col='ead',
        base_col='lgd_base',
        downturn_col='lgd_downturn',
        final_col='lgd_final',
        recovery_col='time_to_sell_base',
    ),
    _summary_from_loan_csv(
        'Bridging Loan',
        'bridging_loan_level_output.csv',
        ead_col='ead',
        base_col='lgd_base',
        downturn_col='lgd_downturn',
        final_col='lgd_final',
        recovery_col='exit_time_base',
    ),
    _summary_from_loan_csv(
        'Mezzanine / 2nd Mortgage',
        'mezz_second_mortgage_loan_level_output.csv',
        ead_col='mezz_ead',
        base_col='lgd_base',
        downturn_col='lgd_downturn',
        final_col='lgd_final',
        recovery_col='time_to_recovery_base',
    ),
]

comparison_df = pd.DataFrame(rows)
comparison_df['downturn_sensitivity_pp'] = (
    comparison_df['ead_weighted_lgd_downturn'] - comparison_df['ead_weighted_lgd_base']
) * 100
comparison_df['downturn_sensitivity_ratio'] = (
    comparison_df['ead_weighted_lgd_downturn'] / comparison_df['ead_weighted_lgd_base']
)
comparison_df['final_minus_base_pp'] = (
    comparison_df['ead_weighted_lgd_final'] - comparison_df['ead_weighted_lgd_base']
) * 100

comparison_df['product'] = pd.Categorical(
    comparison_df['product'],
    categories=PRODUCT_ORDER,
    ordered=True,
)
comparison_df = comparison_df.sort_values('product').reset_index(drop=True)

print('Integrated product rows:', len(comparison_df))
print('Available rows:', int((comparison_df['coverage_status'] == 'available').sum()))
display(
    comparison_df[
        [
            'product',
            'loan_count',
            'total_ead',
            'ead_weighted_lgd_base',
            'ead_weighted_lgd_downturn',
            'ead_weighted_lgd_final',
            'downturn_sensitivity_pp',
            'mean_recovery_time_months',
            'coverage_status',
        ]
    ].round(4)
)


## 3. Compare Weighted LGD, Downturn Sensitivity, and Recovery Time


In [ ]:
available = comparison_df[comparison_df['coverage_status'] == 'available'].copy()

fig, axes = plt.subplots(1, 3, figsize=(22, 6))

plot_lgd = available.melt(
    id_vars='product',
    value_vars=['ead_weighted_lgd_base', 'ead_weighted_lgd_downturn', 'ead_weighted_lgd_final'],
    var_name='metric',
    value_name='weighted_lgd',
)

sns.barplot(data=plot_lgd, x='product', y='weighted_lgd', hue='metric', ax=axes[0])
axes[0].set_title('Weighted LGD by Product (Base / Downturn / Final)')
axes[0].set_xlabel('Product')
axes[0].set_ylabel('EAD-weighted LGD')
axes[0].tick_params(axis='x', rotation=35)

sns.barplot(data=available, x='product', y='downturn_sensitivity_pp', color='#dd8452', ax=axes[1])
axes[1].set_title('Downturn Sensitivity by Product')
axes[1].set_xlabel('Product')
axes[1].set_ylabel('Downturn minus Base (percentage points)')
axes[1].tick_params(axis='x', rotation=35)

sns.barplot(data=available, x='product', y='mean_recovery_time_months', color='#4c72b0', ax=axes[2])
axes[2].set_title('Mean Recovery Time by Product')
axes[2].set_xlabel('Product')
axes[2].set_ylabel('Months')
axes[2].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'cross_product_integrated_comparison.png', dpi=140, bbox_inches='tight')
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)
    print('Plot display skipped (set LGD_NOTEBOOK_SHOW_PLOTS=1 to show).')

comparison_view = available[
    [
        'product',
        'ead_weighted_lgd_base',
        'ead_weighted_lgd_downturn',
        'ead_weighted_lgd_final',
        'downturn_sensitivity_pp',
        'downturn_sensitivity_ratio',
        'mean_recovery_time_months',
        'notes',
    ]
].copy()

comparison_view = comparison_view.sort_values('ead_weighted_lgd_downturn', ascending=False).reset_index(drop=True)
print('Cross-product weighted comparison:')
display(comparison_view.round(4))


## 4. Portfolio Mix Analysis


In [ ]:
mix_df = available.copy()
total_portfolio_ead = mix_df['total_ead'].sum()

mix_df['portfolio_ead_share'] = mix_df['total_ead'] / total_portfolio_ead
mix_df['portfolio_base_lgd_contribution'] = mix_df['portfolio_ead_share'] * mix_df['ead_weighted_lgd_base']
mix_df['portfolio_downturn_lgd_contribution'] = mix_df['portfolio_ead_share'] * mix_df['ead_weighted_lgd_downturn']
mix_df['portfolio_final_lgd_contribution'] = mix_df['portfolio_ead_share'] * mix_df['ead_weighted_lgd_final']

portfolio_summary = pd.DataFrame([
    {
        'portfolio_total_ead': total_portfolio_ead,
        'portfolio_weighted_lgd_base': mix_df['portfolio_base_lgd_contribution'].sum(),
        'portfolio_weighted_lgd_downturn': mix_df['portfolio_downturn_lgd_contribution'].sum(),
        'portfolio_weighted_lgd_final': mix_df['portfolio_final_lgd_contribution'].sum(),
        'products_with_available_metrics': len(mix_df),
    }
])

print('Portfolio integrated summary:')
display(portfolio_summary.round(6))

mix_view = mix_df[
    [
        'product',
        'total_ead',
        'portfolio_ead_share',
        'ead_weighted_lgd_final',
        'portfolio_final_lgd_contribution',
    ]
].sort_values('portfolio_ead_share', ascending=False).reset_index(drop=True)

display(mix_view.round(4))

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(mix_view['product'], mix_view['portfolio_ead_share'], color='#55a868')
ax.invert_yaxis()
ax.set_title('Portfolio Mix by Product (EAD Share)')
ax.set_xlabel('Share of Portfolio EAD')
ax.set_ylabel('Product')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'cross_product_portfolio_mix.png', dpi=140, bbox_inches='tight')
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)


## 5. Cross-Product Risk Ranking


In [ ]:
risk_df = available.copy()

risk_df['rank_lgd_level'] = risk_df['ead_weighted_lgd_downturn'].rank(ascending=False, method='dense')
risk_df['rank_downturn_sensitivity'] = risk_df['downturn_sensitivity_pp'].rank(ascending=False, method='dense')
risk_df['rank_recovery_time'] = risk_df['mean_recovery_time_months'].rank(ascending=False, method='dense')

rank_cols = ['rank_lgd_level', 'rank_downturn_sensitivity', 'rank_recovery_time']
risk_df['composite_risk_score'] = risk_df[rank_cols].mean(axis=1, skipna=True)
risk_df['risk_rank'] = risk_df['composite_risk_score'].rank(ascending=True, method='dense')

risk_ranking = risk_df[
    [
        'risk_rank',
        'product',
        'ead_weighted_lgd_downturn',
        'downturn_sensitivity_pp',
        'mean_recovery_time_months',
        'composite_risk_score',
        'rank_lgd_level',
        'rank_downturn_sensitivity',
        'rank_recovery_time',
    ]
].sort_values(['risk_rank', 'ead_weighted_lgd_downturn'], ascending=[True, False]).reset_index(drop=True)

print('Risk ranking (higher rank = higher relative risk in this proxy framework):')
display(risk_ranking.round(4))


## Assumptions and Limitations

- This integrated comparison uses synthetic data and transparent proxy assumptions.
- Non-core secured products now export explicit `lgd_final` outputs (rather than using downturn as a proxy) for more apples-to-apples comparison.
- CRE now exports explicit `time_to_recovery_months_base`, defined as expected months from default to economic resolution.
- Out-of-time and vintage validation in this repo is simulated and should be recalibrated with real internal workout datasets before production use.


In [ ]:
# Export cross-product outputs
comparison_df.to_csv(TABLE_DIR / 'cross_product_comparison.csv', index=False)
mix_view.to_csv(TABLE_DIR / 'cross_product_portfolio_mix.csv', index=False)
risk_ranking.to_csv(TABLE_DIR / 'cross_product_risk_ranking.csv', index=False)
portfolio_summary.to_csv(TABLE_DIR / 'cross_product_portfolio_summary.csv', index=False)

print('Saved cross-product outputs:')
print('- cross_product_segmentation_table.csv')
print('- cross_product_comparison.csv')
print('- cross_product_portfolio_mix.csv')
print('- cross_product_risk_ranking.csv')
print('- cross_product_portfolio_summary.csv')


---

## APS 113 Calibration Comparison — Cross-Product

> **SYNTHETIC HISTORICAL CALIBRATION DATA — FOR DEMONSTRATION ONLY**
>
> This section overlays the APS 113 calibration results from notebooks 02-12
> onto the proxy-based comparison above. It shows:
> calibrated LGD waterfall, seniority rank-ordering, MoC summary,
> APRA benchmark comparison, and indicative RWA impact.

**Correct APS 113 pipeline order per product:**
Realised LGD → Long-run LGD (vintage-EWA) → Downturn overlay →
Frye-Jacobs correlation adj → MoC (s.63-65) → Regulatory floor (s.58) → Final


In [ ]:
# ── Load per-product calibration adjustment summaries (from notebooks 02-12) ─
import os, sys
from pathlib import Path
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

OUTPUTS = Path('..') / 'outputs' / 'tables'
PRODUCTS = [
    'mortgage', 'commercial_cashflow', 'receivables', 'trade_contingent',
    'asset_equipment', 'development_finance', 'cre_investment',
    'residual_stock', 'land_subdivision', 'bridging', 'mezz_second_mortgage',
]


def _load_cal_adj(product: str) -> dict | None:
    path = OUTPUTS / f'{product}_calibration_adjustments.csv'
    if not path.exists():
        return None
    row = pd.read_csv(path).iloc[0].to_dict()
    row['product'] = product
    return row


def _load_moc(product: str) -> pd.DataFrame | None:
    path = OUTPUTS / f'{product}_moc_register.csv'
    if not path.exists():
        return None
    df = pd.read_csv(path)
    df['product'] = product
    return df


rows = [r for p in PRODUCTS if (r := _load_cal_adj(p)) is not None]
cal_df = pd.DataFrame(rows)
moc_frames = [f for p in PRODUCTS if (f := _load_moc(p)) is not None]
moc_all = pd.concat(moc_frames, ignore_index=True) if moc_frames else pd.DataFrame()

if cal_df.empty:
    print("No calibration outputs found. Run notebooks 02-12 calibration sections first.")
    print("Or: python scripts/run_calibration_pipeline.py --products all")
else:
    print(f"Loaded calibration results for {len(cal_df)} products")
    cols = ['ewa_realised_lgd', 'ewa_long_run_lgd', 'ewa_downturn_lgd',
            'ewa_lgd_with_moc', 'ewa_lgd_final']
    display(cal_df.set_index('product')[[c for c in cols if c in cal_df.columns]].round(4))


In [ ]:
# ── Calibration waterfall: all products ─────────────────────────────────────
if not cal_df.empty:
    numeric_cols = ['ewa_realised_lgd', 'ewa_long_run_lgd', 'ewa_downturn_lgd',
                    'ewa_lgd_with_moc', 'ewa_lgd_final']
    stage_labels = ['Realised\n(2014-24)', 'Long-run\n(EWA)',
                    'Downturn', '+ MoC', 'Final\n(Floor)']
    available_cols = [c for c in numeric_cols if c in cal_df.columns]
    n_stages = len(available_cols)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    # Left: grouped bar chart — all pipeline stages per product
    x = np.arange(len(cal_df))
    width = 0.14
    colors = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c', '#8e44ad']
    for i, (col, label) in enumerate(zip(available_cols, stage_labels[:n_stages])):
        axes[0].bar(x + i * width, cal_df[col], width, label=label,
                    color=colors[i % len(colors)], alpha=0.85)
    axes[0].set_xticks(x + width * (n_stages - 1) / 2)
    axes[0].set_xticklabels(cal_df['product'].str.replace('_', '\n'), fontsize=7)
    axes[0].set_ylabel('EAD-Weighted LGD')
    axes[0].set_title('APS 113 Calibration Waterfall — All Products')
    axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    axes[0].legend(fontsize=8)

    # Right: final calibrated LGD rank order
    sorted_df = cal_df.sort_values('ewa_lgd_final', ascending=True)
    bars = axes[1].barh(sorted_df['product'], sorted_df['ewa_lgd_final'],
                        color='#8e44ad', alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, sorted_df['ewa_lgd_final']):
        axes[1].text(val + 0.003, bar.get_y() + bar.get_height() / 2,
                     f'{val:.1%}', va='center', fontsize=9)
    axes[1].set_xlabel('Final Calibrated LGD (EWA)')
    axes[1].set_title('Final LGD Rank Order (APS 113 s.58 floor applied)')
    axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))

    plt.tight_layout()
    plt.savefig(Path('..') / 'outputs' / 'figures' / 'cross_product_calibration_waterfall.png',
                dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# ── Seniority rank-ordering check (APS 113 best practice) ──────────────────
# Expected: more senior exposures should generally show lower LGD.
# This validates rank-ordering reasonableness across the product hierarchy.
SENIORITY_ORDER = {
    'mortgage': 1, 'cre_investment': 2, 'asset_equipment': 3,
    'commercial_cashflow': 4, 'development_finance': 5, 'receivables': 6,
    'trade_contingent': 7, 'land_subdivision': 8, 'residual_stock': 9,
    'bridging': 10, 'mezz_second_mortgage': 11,
}

if not cal_df.empty and 'ewa_lgd_final' in cal_df.columns:
    from scipy.stats import spearmanr

    rank_df = cal_df[['product', 'ewa_lgd_final']].copy()
    rank_df['seniority_rank'] = rank_df['product'].map(SENIORITY_ORDER)
    rank_df = rank_df.dropna(subset=['seniority_rank']).sort_values('seniority_rank')
    rank_df['lgd_rank'] = rank_df['ewa_lgd_final'].rank()
    rank_df['rank_diff'] = rank_df['lgd_rank'] - rank_df['seniority_rank']
    rank_df['rank_ordering_ok'] = rank_df['rank_diff'].abs() <= 2

    rho, pval = spearmanr(rank_df['seniority_rank'], rank_df['lgd_rank'])

    print('=== Seniority Rank-Ordering Check ===')
    print(f'  Spearman rho (seniority vs LGD rank): {rho:.3f}  p={pval:.3f}')
    print(f'  Expected: positive correlation (senior = first-ranking = lower LGD)')
    violations = (~rank_df['rank_ordering_ok']).sum()
    print(f'  Rank violations (|diff| > 2): {violations}')
    if violations == 0:
        print('  PASS — rank ordering consistent with seniority hierarchy')
    else:
        print('  REVIEW — investigate rank violations against model assumptions')
    display(rank_df[['product', 'seniority_rank', 'lgd_rank', 'ewa_lgd_final',
                      'rank_diff', 'rank_ordering_ok']].round(3))


In [ ]:
# ── MoC summary across all products (APS 113 s.63-65) ──────────────────────
if not moc_all.empty:
    print('=== MoC Summary — All Products ===')
    moc_components = [c for c in moc_all.columns
                      if c.endswith('_moc') and c != 'total_moc']
    if 'total_moc' in moc_all.columns:
        moc_summary = (
            moc_all.groupby('product')[['total_moc'] + moc_components]
            .mean()
            .round(4)
        )
        display(moc_summary)

        if moc_components:
            fig, ax = plt.subplots(figsize=(13, 5))
            moc_pivot = moc_all.groupby('product')[moc_components].mean()
            moc_pivot.plot(kind='bar', stacked=True, ax=ax,
                           colormap='tab10', edgecolor='white', width=0.7)
            ax.set_ylabel('MoC Add-on (absolute LGD)')
            ax.set_title('APS 113 s.65 MoC Components by Product — Five Required Sources')
            ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
            ax.legend(loc='upper right', fontsize=8)
            plt.xticks(rotation=30, ha='right')
            plt.tight_layout()
            plt.savefig(Path('..') / 'outputs' / 'figures' / 'cross_product_moc_summary.png',
                        dpi=150, bbox_inches='tight')
            plt.show()
    moc_all.to_csv(OUTPUTS / 'moc_summary_all_products.csv', index=False)
    print('Saved outputs/tables/moc_summary_all_products.csv')
else:
    print('MoC registers not generated yet — run notebooks 02-12 calibration sections')


In [ ]:
# ── APRA ADI benchmark comparison ───────────────────────────────────────────
apra_path = OUTPUTS / 'apra_benchmark_comparison.csv'
if apra_path.exists():
    apra_df = pd.read_csv(apra_path)
    print('=== APRA ADI Benchmark Comparison ===')
    print('NOTE: APRA impairment ratio is a directional proxy only — not a direct LGD benchmark.')
    display(apra_df.round(4))
else:
    print('APRA benchmark not generated — run: python scripts/run_calibration_pipeline.py')

# ── Indicative RWA impact: proxy vs calibrated LGD ─────────────────────────
if not cal_df.empty and 'ewa_lgd_final' in cal_df.columns:

    def _irb_rwa_approx(ead: float, pd_: float, lgd: float) -> float:
        # Simplified IRB corporate RWA. APS 113 Attachment C.
        r = (0.12 * (1 - np.exp(-50 * pd_)) / (1 - np.exp(-50))
             + 0.24 * (1 - (1 - np.exp(-50 * pd_)) / (1 - np.exp(-50))))
        z = (pd_ * 1.06 + (pd_ * (1 - pd_)) ** 0.5 * 1.645
             * (r ** 0.5 / (1 - r) ** 0.5 - pd_ * 0.5))
        k = max(lgd * z, 0.0)
        return k * ead * 12.5

    rwa_df = cal_df[['product']].copy()
    rwa_df['proxy_lgd'] = cal_df.get('ewa_realised_lgd',
                                      cal_df['ewa_lgd_final'] * 0.85)
    rwa_df['calibrated_lgd'] = cal_df['ewa_lgd_final']
    PD = 0.02
    EAD = 100_000_000  # $100m illustrative

    rwa_df['rwa_proxy'] = rwa_df['proxy_lgd'].apply(
        lambda lgd: _irb_rwa_approx(EAD, PD, lgd))
    rwa_df['rwa_calibrated'] = rwa_df['calibrated_lgd'].apply(
        lambda lgd: _irb_rwa_approx(EAD, PD, lgd))
    rwa_df['rwa_delta_pct'] = (
        (rwa_df['rwa_calibrated'] - rwa_df['rwa_proxy']) / rwa_df['rwa_proxy']
    )

    print('\n=== Indicative RWA Impact: Proxy vs Calibrated LGD ===')
    print('(Illustrative: $100m EAD per product, flat PD=2%, simplified IRB formula)')
    display(rwa_df.set_index('product')[
        ['proxy_lgd', 'calibrated_lgd', 'rwa_proxy', 'rwa_calibrated', 'rwa_delta_pct']
    ].round(4))
    rwa_df.to_csv(OUTPUTS / 'cross_product_rwa_impact.csv', index=False)
    print('Saved outputs/tables/cross_product_rwa_impact.csv')
